In [ ]:

import pandas as pd

df = pd.read_excel("/content/veda_day22_cohort_retention_dataset.xlsx")

print("Dataset shape:", df.shape)
print("Unique customers:", df["customer_id"].nunique())
display(df.head())
print(df.isnull().sum())

In [ ]:

# Convert date columns to datetime format
df["signup_date"] = pd.to_datetime(df["signup_date"])
df["order_date"] = pd.to_datetime(df["order_date"])

# Create signup month cohort
df["cohort_month"] = (
    df["signup_date"].dt.to_period("M")
)

# Create order activity month
df["order_month"] = (
    df["order_date"].dt.to_period("M")
)

# Display results
display(df[[
    "customer_id",
    "signup_date",
    "order_date",
    "cohort_month",
    "order_month"
]].head(10))

In [ ]:

# Step 1: Calculate months since signup
df["cohort_index"] = (
    (df["order_month"].dt.year - df["cohort_month"].dt.year) * 12
    + (df["order_month"].dt.month - df["cohort_month"].dt.month)
)

# Step 2: Count unique active customers
cohort_data = df.groupby(
    ["cohort_month", "cohort_index"]
)["customer_id"].nunique().reset_index()

# Step 3: Create cohort table
cohort_table = cohort_data.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="customer_id"
)

# Step 4: Calculate original cohort size
cohort_size = df.groupby(
    "cohort_month"
)["customer_id"].nunique()

# Step 5: Calculate retention percentage
retention_table = cohort_table.div(
    cohort_size, axis=0
) * 100

print("COHORT CUSTOMER TABLE")
display(cohort_table)

print("RETENTION TABLE (%)")
display(retention_table.round(2))

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.heatmap(
    retention_table,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    vmin=0,
    vmax=100,
    linewidths=0.5
)

plt.title("Monthly Cohort Retention Heatmap")
plt.xlabel("Months Since Signup")
plt.ylabel("Signup Cohort Month")
plt.tight_layout()
plt.show()

In [8]:

import sqlite3
import pandas as pd

# Create a temporary SQL database
conn = sqlite3.connect(":memory:")

# Prepare the dataset for SQL
sql_df = df.copy()
sql_df["cohort_month"] = sql_df["cohort_month"].astype(str)
sql_df["order_month"] = sql_df["order_month"].astype(str)

# Create the orders table
sql_df.to_sql("orders", conn, if_exists="replace", index=False)

print("SQL database and orders table created successfully!")

SQL database and orders table created successfully!


In [ ]:

# SQL Query 1: Count unique customers in each signup cohort

query1 = """
SELECT
    cohort_month,
    COUNT(DISTINCT customer_id) AS cohort_size
FROM orders
GROUP BY cohort_month
ORDER BY cohort_month;
"""

# Execute the SQL query using Python
result1 = pd.read_sql_query(query1, conn)

# Display the result
print("COHORT SIZE ANALYSIS")
display(result1)

In [ ]:

# SQL Query 2: Monthly Active Customers by Cohort

query2 = """
SELECT
    cohort_month,
    order_month,
    COUNT(DISTINCT customer_id) AS active_customers
FROM orders
GROUP BY
    cohort_month,
    order_month
ORDER BY
    cohort_month,
    order_month;
"""

# Execute SQL query
result2 = pd.read_sql_query(query2, conn)

# Display results
print("MONTHLY ACTIVE CUSTOMERS BY COHORT")
display(result2)

In [ ]:

# SQL Query 3: Calculate total revenue by signup cohort

query3 = """
SELECT
    cohort_month,
    SUM(order_amount) AS total_revenue
FROM orders
GROUP BY cohort_month
ORDER BY cohort_month;
"""

# Execute SQL query
result3 = pd.read_sql_query(query3, conn)

print("TOTAL REVENUE BY COHORT")
display(result3)

In [ ]:

print("QUERY 1: COHORT SIZE")
display(result1)

print("\nQUERY 2: MONTHLY ACTIVE CUSTOMERS BY COHORT")
display(result2)

print("\nQUERY 3: TOTAL REVENUE BY COHORT")
display(result3)

print("\nSQL analysis completed successfully!")


In [ ]:
# BUSINESS INSIGHTS: COHORT RETENTION ANALYSIS

print("BUSINESS INSIGHTS")
print("=" * 45)

# 1. Customer count by signup cohort
print("\n1. CUSTOMER COUNT BY SIGNUP COHORT")
display(result1)

# 2. Monthly customer retention percentage
print("\n2. MONTHLY CUSTOMER RETENTION (%)")
display(retention_table.round(2))

# 3. Revenue by signup cohort
print("\n3. TOTAL REVENUE BY COHORT")
display(result3)

# 4. Identify the highest-revenue cohort
highest_revenue = result3.loc[
    result3["total_revenue"].idxmax()
]

print("\n4. HIGHEST REVENUE COHORT")
print("Cohort Month:", highest_revenue["cohort_month"])
print("Total Revenue:", highest_revenue["total_revenue"])

print("\n" + "=" * 45)
print("COHORT RETENTION ANALYSIS COMPLETED!")